In [ ]:
!pip install qiskit qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 7.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.2 MB/s  0:00:00
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [qiskit-ibm-runtime]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.46.2 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 3.0.0 which is incompatible.
ibis-fra

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from numpy import pi

def build_circuit(state, version, include_measurements=True):
    """
    Builds the 5-qubit Figure 6 superdense coding automated error-correction circuit.
    states: '00' (I), '01' (Z), '10' (X), '11' (XZ) -> maps to c[1]c[0]
    versions: 'base', 'opt_barriers', 'opt_no_barriers'
    """
    q = QuantumRegister(5, 'q')
    c = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(q, c, name=f"{state}_{version}")

    def add_barrier():
        if version != 'opt_no_barriers':
            qc.barrier(q)

    if version == 'base':
        # 1. Entanglement
        qc.h(q[0])
        qc.cx(q[0], q[1])
        add_barrier()

        # 2. Alice's Message Encoding (c[1] = Bit-flip X, c[0] = Phase-flip Z)
        if state in ['10', '11']:
            qc.x(q[1])
        if state in ['01', '11']:
            qc.z(q[1])
        add_barrier()

        # Initial Parity & Flag Setup
        qc.cx(q[1], q[2])
        qc.h(q[3])
        qc.cx(q[3], q[1])
        add_barrier()

        # 3. Combined Channel Error Injection (Figure 6: X, Z, and Rz(pi/3) on q[1])
        qc.x(q[1])
        qc.z(q[1])
        qc.rz(pi / 3, q[1])
        add_barrier()

        # 4. Syndrome Extraction
        qc.cx(q[0], q[2])
        qc.cx(q[3], q[0])
        qc.h(q[3])
        add_barrier()

        qc.h(q[4])
        qc.cx(q[4], q[0])

        # 5. Automated Error Correction
        qc.cx(q[4], q[1])
        qc.cx(q[1], q[4])
        add_barrier()

        qc.h(q[3])
        qc.cx(q[3], q[0])
        qc.cx(q[3], q[1])
        qc.h(q[3])
        qc.cz(q[1], q[3])
        add_barrier()

        # 6. Bell Basis Decoding
        qc.cx(q[0], q[2])
        qc.cx(q[1], q[2])
        qc.cx(q[2], q[1])
        add_barrier()

        qc.cx(q[0], q[1])
        qc.h(q[0])

    else:
        # OPTIMIZED (Hardware-Aware Native Pulses & Commutation Reduction)
        # 1. Entanglement
        qc.ry(pi / 2, q[0])
        qc.cx(q[0], q[1])
        add_barrier()

        # 2. Alice's Hardware-Aware Encoding (Native SX & 0ns Virtual Rz)
        if state in ['10', '11']:
            qc.sx(q[1])
            qc.sx(q[1])
        if state in ['01', '11']:
            qc.rz(pi, q[1])
        add_barrier()

        # Initial Parity & Flag Setup
        qc.cx(q[1], q[2])
        qc.ry(pi / 2, q[3])
        qc.cx(q[3], q[1])
        add_barrier()

        # 3. Combined Channel Error Injection (Native SX & Merged Virtual Rz)
        qc.sx(q[1])
        qc.sx(q[1])
        qc.rz(pi, q[1])
        qc.rz(pi / 3, q[1])
        add_barrier()

        # 4. Syndrome Extraction (Redundant cx(q3,q0) and h(q3) pairs cancelled)
        qc.cx(q[0], q[2])
        add_barrier()

        qc.ry(pi / 2, q[4])
        qc.cx(q[4], q[0])

        # 5. Automated Error Correction
        qc.cx(q[4], q[1])
        qc.cx(q[1], q[4])
        add_barrier()

        qc.cx(q[3], q[1])
        qc.ry(-pi / 2, q[3])  # 0-degree inverse basis close
        qc.cz(q[1], q[3])
        add_barrier()

        # 6. Bell Basis Decoding
        qc.cx(q[0], q[2])
        qc.cx(q[1], q[2])
        qc.cx(q[2], q[1])
        add_barrier()

        qc.cx(q[0], q[1])
        qc.ry(-pi / 2, q[0])  # 0-degree inverse decoding

    if include_measurements:
        qc.measure(q[0], c[0])
        qc.measure(q[1], c[1])

    return qc

def verify_ideal_output(state, version):
    """Verifies that the circuit deterministically outputs the target 2-bit state."""
    qc_no_meas = build_circuit(state, version, include_measurements=False)
    sv = Statevector.from_instruction(qc_no_meas)
    probs = sv.probabilities_dict([0, 1])  # Marginal probabilities on (q[0], q[1]) -> 'c[1]c[0]'
    return probs.get(state, 0.0) * 100

def extract_metrics(tp_circ, backend_target):
    depth = tp_circ.depth()
    ops = tp_circ.count_ops()
    two_qubit_gates = ops.get('ecr', 0) + ops.get('cx', 0) + ops.get('cz', 0)

    try:
        duration_s = tp_circ.estimate_duration(backend_target, unit="s")
        duration_ns = duration_s * 1e9
    except AttributeError:
        duration_ns = 0

    return depth, two_qubit_gates, duration_ns
if __name__ == "__main__":
    print("Connecting to IBM Quantum...")
    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token="YOUR_IBM_QUANTUM_TOKEN",
        instance="YOUR_IBM_QUANTUM_INSTANCE"
    )
    backend = service.backend("ibm_kingston")
    states = ['00', '01', '10', '11']

    # Separate the untranspiled circuit (ref_qc) from the transpiled circuit (ref_base)
    # so we look up the 5 original virtual qubits in initial_layout
    ref_qc = build_circuit('11', 'base')
    ref_base = transpile(ref_qc, backend, scheduling_method='alap', optimization_level=3, seed_transpiler=42)
    fixed_layout = [ref_base.layout.initial_layout[q] for q in ref_qc.qubits]
    print(f"Connected to {backend.name}. Locked Physical Layout: {fixed_layout}")

    for state in states:
        p_base = verify_ideal_output(state, 'base')
        p_opt = verify_ideal_output(state, 'opt_no_barriers')

        print("\n" + "="*90)
        print(f"  STATE ENCODING: {state}  |  Ideal Target Fidelity: Base = {p_base:.1f}%, Opt = {p_opt:.1f}%")
        print("="*90)
        print(f"{'Metric':<25} | {'Base Circuit':<15} | {'Opt (With Barriers)':<20} | {'Opt (No Barriers)'}")
        print("-" * 90)

        # 1. Build circuits dynamically
        qc_base = build_circuit(state, 'base')
        qc_opt_b = build_circuit(state, 'opt_barriers')
        qc_opt_nb = build_circuit(state, 'opt_no_barriers')

        # 2. Transpile all versions onto the identical physical layout
        tp_base = transpile(qc_base, backend, scheduling_method='alap', optimization_level=3, initial_layout=fixed_layout, seed_transpiler=42)
        tp_opt_b = transpile(qc_opt_b, backend, scheduling_method='alap', optimization_level=3, initial_layout=fixed_layout, seed_transpiler=42)
        tp_opt_nb = transpile(qc_opt_nb, backend, scheduling_method='alap', optimization_level=3, initial_layout=fixed_layout, seed_transpiler=42)

        # 3. Extract Metrics
        d_base, g_base, t_base = extract_metrics(tp_base, backend.target)
        d_opt_b, g_opt_b, t_opt_b = extract_metrics(tp_opt_b, backend.target)
        d_opt_nb, g_opt_nb, t_opt_nb = extract_metrics(tp_opt_nb, backend.target)

        # 4. Print Row Comparisons
        print(f"{'Circuit Depth':<25} | {d_base:<15} | {d_opt_b:<20} | {d_opt_nb}")
        print(f"{'2-Qubit Gate Count':<25} | {g_base:<15} | {g_opt_b:<20} | {g_opt_nb}")
        print(f"{'Execution Latency (ns)':<25} | {t_base:<15.2f} | {t_opt_b:<20.2f} | {t_opt_nb:.2f}")
        print("-" * 90)


qiskit_runtime_service._discover_account:WARNING:2026-10-29 16:36:04,719: Loading account with the given token. A saved account will not be used.


Connecting to IBM Quantum...
Connected to ibm_kingston. Locked Physical Layout: [152, 151, 150, 138, 153]

  STATE ENCODING: 00  |  Ideal Target Fidelity: Base = 100.0%, Opt = 100.0%
Metric                    | Base Circuit    | Opt (With Barriers)  | Opt (No Barriers)
------------------------------------------------------------------------------------------
Circuit Depth             | 92              | 83                   | 65
2-Qubit Gate Count        | 31              | 29                   | 25
Execution Latency (ns)    | 5172.00         | 4712.00              | 4416.00
------------------------------------------------------------------------------------------

  STATE ENCODING: 01  |  Ideal Target Fidelity: Base = 100.0%, Opt = 100.0%
Metric                    | Base Circuit    | Opt (With Barriers)  | Opt (No Barriers)
------------------------------------------------------------------------------------------
Circuit Depth             | 93              | 84                   | 66
